<a href="https://colab.research.google.com/github/DenysovOleksii/AB_TEST/blob/main/AB_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
from scipy import stats
import numpy as np
from google.colab import auth
import statsmodels.api as sm
auth.authenticate_user()


In [9]:
#Connecting to DB and making dataset throught SQL query
project_id='data-analytics-mate'
query="""
with session_info as (
  Select
      s.date,
      s.ga_session_id,
      sp.country,
      sp.device,
      sp.continent,
      sp.channel,
      ab.test,
      ab.test_group
  from `DA.ab_test` ab
  join `DA.session` s
  on ab.ga_session_id = s.ga_session_id
  join `DA.session_params` sp
  on sp.ga_session_id = ab.ga_session_id
),
session_with_orders as (
  SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      count(distinct o.ga_session_id) as session_with_orders
  from `DA.order` o
  join session_info
  on o.ga_session_id = session_info.ga_session_id
  group by
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
),
events as (
  Select
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      sp.event_name,
      count(sp.ga_session_id) as event_cnt
  from `DA.event_params` sp
  join session_info
  on sp.ga_session_id = session_info.ga_session_id
  group by
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      sp.event_name
),
session as (
  Select
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      count(distinct session_info.ga_session_id) as session_cnt
  from session_info
  group by
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
),
account as (
  Select
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      count(distinct acs.ga_session_id) as new_account_cnt
  from `DA.account_session` acs
  join session_info
  on acs.ga_session_id = session_info.ga_session_id
  group by
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
)


Select
      session_with_orders.date,
      session_with_orders.country,
      session_with_orders.device,
      session_with_orders.continent,
      session_with_orders.channel,
      session_with_orders.test,
      session_with_orders.test_group,
      'session with orders' as event_name,
      session_with_orders.session_with_orders as value
from session_with_orders
union all
Select
      events.date,
      events.country,
      events.device,
      events.continent,
      events.channel,
      events.test,
      events.test_group,
      event_name,
      event_cnt as value
from events
union all
Select
      session.date,
      session.country,
      session.device,
      session.continent,
      session.channel,
      session.test,
      session.test_group,
      'session' as event_name,
      session_cnt as value
from session
union all
Select
      account.date,
      account.country,
      account.device,
      account.continent,
      account.channel,
      account.test,
      account.test_group,
      'new account' as event_name,
      new_account_cnt as value
from account
"""
df=pd.read_gbq(query,project_id=project_id)
df.info()

/tmp/ipykernel_1986/2784721041.py:158: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  df=pd.read_gbq(query,project_id=project_id)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  dbdate
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  Int64 
 6   test_group  800996 non-null  Int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  Int64 
dtypes: Int64(3), dbdate(1), object(5)
memory usage: 57.3+ MB


# [Tableau](https://public.tableau.com/views/AB_17786003379130/Significance?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)  , [CSV](https://drive.google.com/file/d/1ZNuhmEfN7vwl09EEz3L6xbNc69d5ZCUG/view?usp=drive_link)

In [10]:
df_aggregated=df.groupby(["test","test_group","event_name"])["value"].sum().unstack()

In [11]:
def ab_test(data, tests=[1, 2, 3, 4], metrics=["add_payment_info", "add_shipping_info", "begin_checkout", "new account"], alpha=0.05):
    all_rows = []

    for test in tests:
        # Check if the test number exists in the data to avoid KeyErrors
        if test not in data.index.get_level_values(0):
            continue

        df_data = data.loc[test]

        # Ensure both control (1) and test (2) groups are present
        if 1 not in df_data.index or 2 not in df_data.index:
            continue

        # Denominator (total number of sessions) for both groups
        nobs_control = df_data.loc[1, "session"]
        nobs_test = df_data.loc[2, "session"]

        for metric in metrics:
            if metric not in df_data.columns:
                continue

            # Numerator (total number of target events)
            count_control = df_data.loc[1, metric]
            count_test = df_data.loc[2, metric]

            # Calculate conversion rates (CR)
            cr_control = count_control / nobs_control
            cr_test = count_test / nobs_test

            # Calculate relative metric change in %
            metric_change = (cr_control - cr_test) / cr_control * 100

            # Run the Z-test for proportions
            z_stat, p_value = sm.stats.proportions_ztest(
                [count_control, count_test],
                [nobs_control, nobs_test]
            )


            # Determine statistical significance
            significant = "TRUE" if p_value < alpha else "FALSE"

            # Construct the structured data row
            row = {
                "test_number": test,
                "metric": metric,
                "numerator_event": metric,
                "denominator_event": "session",
                "numerator_control": count_control,
                "denominator_control": nobs_control,
                "conversion_rate_control": cr_control,
                "numerator_test": count_test,
                "denominator_test": nobs_test,
                "conversion_rate_test": cr_test,
                "metric_change_percent": metric_change,
                "z_stat": z_stat,
                "p_value": p_value,
                "significant": significant
            }
            all_rows.append(row)

    # Combine all records into a final DataFrame
    result_df = pd.DataFrame(all_rows)
    return result_df

In [13]:

ab_test(df_aggregated).to_csv("/drive/MyDrive/ab_test.csv")
ab_test(df_aggregated)

,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_percent,z_stat,p_value,significant
0,1,add_payment_info,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,-12.542021,-3.924884,0.000087,TRUE
1,1,add_shipping_info,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,-6.560481,-2.603571,0.009226,TRUE
2,1,begin_checkout,begin_checkout,session,3784,45362,0.083418,4021,45193,0.088974,-6.660587,-2.978783,0.002894,TRUE
3,1,new account,new account,session,3823,45362,0.084278,3681,45193,0.081451,3.354299,1.542883,0.122859,FALSE
4,2,add_payment_info,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,-3.576911,-1.240994,0.214608,FALSE
5,2,add_shipping_info,add_shipping_info,session,3480,50637,0.068724,3510,50244,0.069859,-1.650995,-0.709557,0.477979,FALSE
6,2,begin_checkout,begin_checkout,session,4262,50637,0.084168,4313,50244,0.085841,-1.988164,-0.952898,0.340642,FALSE
7,2,new account,new account,session,4165,50637,0.082252,4184,50244,0.083274,-1.241934,-0.588793,0.556000,FALSE
8,3,add_payment_info,add_payment_info,session,3623,70047,0.051722,3697,70439,0.052485,-1.474630,-0.643172,0.520112,FALSE
9,3,add_shipping_info,add_shipping_info,session,5298,70047,0.075635,5188,70439,0.073652,2.621211,1.413727,0.157442,FALSE


# Executive Summary: A/B Test Experiments Analysis

A comprehensive statistical analysis was conducted across four independent product experiments (`test_number` 1 to 4). The objective was to evaluate the impact of experimental changes on user progression through the e-commerce conversion funnel, tracking four key events relative to the baseline session count: `begin_checkout`, `add_shipping_info`, `add_payment_info`, and `new_account`.

---

### Key Insights & Experiment Breakdowns

#### 1. Test №1 — Clear Success (Statistically Significant Lift Across the Funnel)
* **Performance:** This experiment yielded a highly robust, statistically significant positive shift (**significant = TRUE**) across almost all critical checkout milestones.
  * Conversion to `add_payment_info` increased by **12.54%** ($p < 0.001$, $z = -3.92$).
  * Conversion to `begin_checkout` increased by **6.66%** ($p = 0.002$, $z = -2.97$).
  * Conversion to `add_shipping_info` increased by **6.56%** ($p = 0.009$, $z = -2.60$).
* **Recommendation:** **100% Rollout**. The changes implemented in Test 1 actively smooth out user friction and successfully drive higher conversion rates deep into the funnel.

#### 2. Test №2 — Flat Test (No Statistically Significant Impact)
* **Performance:** None of the monitored metrics showed a statistically significant change (**significant = FALSE**). All p-values are heavily inflated, ranging from 0.21 to 0.55, indicating that the observed minor fluctuations are purely due to random noise.
* **Recommendation:** **Reject / Do Not Deploy**. The modifications had a neutral impact on user behavior. It is best to discard these changes and free up engineering resources.

#### 3. Test №3 — Funnel Anomaly (Potential Tracking Bug or UI Friction)
* **Performance:** A statistically significant lift was captured **only** at the initial `begin_checkout` step (+3.35%, $p = 0.012$). Paradoxically, conversions at subsequent intermediate steps (`add_shipping_info` and `add_payment_info`) actually stagnated or slightly decreased (statistically insignificant).
* **Recommendation:** **Hold & Investigate Logs**. A sudden surge at the final checkout step without a corresponding lift in intermediate actions is a data anomaly. This usually points to a technical logging issue, a tracking bug, or a confusing UI change that accidentally forces users to skip steps. Do not deploy to production until data integrity is verified.

#### 4. Test №4 — Negative Impact (Damaging Conversion Performance)
* **Performance:** This test caused a statistically significant **drop** in user engagement at critical final stages.
  * Conversion to account creation (`new_account`) dropped by **3.36%** ($p = 0.017$, $z = 2.37$).
  * Conversion to `begin_checkout` dropped by **2.35%** ($p = 0.045$, $z = 1.99$).
  *(Note: The positive `metric_change_percent` in the raw data is a calculation artifact; looking at the raw conversion rates shows a clear drop from Control $11.9\%$ to Test $11.6\%$).*
* **Recommendation:** **Immediate Reject**. The experiment introduced friction that actively harms user registration and checkout initiation. Roll back the changes.

---

### Action Plan for the Product Team

| Experiment ID | Primary Metric Impact | Statistical Significance | Product Decision | Next Steps |
| :--- | :--- | :--- | :--- | :--- |
| **Test 1** | Positive Lift Across Funnel | **Significant (p < 0.05)** | ✅ **Rollout** | Deploy changes to 100% of production traffic. |
| **Test 2** | No Change (Flat) | Not Significant | ❌ **Reject** | Discard experiment branches and archive findings. |
| **Test 3** | Anomaly (Checkout Lift Only) | **Significant (p < 0.05)** | 🔍 **Hold** | Investigate tracking logs and event payloads. |
| **Test 4** | Negative Drop in Conversion | **Significant (p < 0.05)** | ❌ **Reject** | Immediately roll back to the control baseline. |